# Closing H1's Last Two Gaps: Standard-CNN Baseline + Noise-Robustness Test

Execution notebook built directly from `h1_completion_plan.docx` Section 8's
run order (Streams A and B). **Nothing in this notebook has been run yet** --
every cell below is ready to execute, but no number in it is real until you
actually run it against this project's real windowed N-CMAPSS data on Drive.

Source of truth for every function here:
- Cells marked **[verbatim from h1_completion_plan.docx]** are copied directly
  from that document's own code blocks.
- Cells marked **[new for this notebook]** fill a gap the plan named but did
  not give literal source for (mainly `benchmark_variant`) -- written to match
  the documented methodology, but not a quote from any document.
- `train_one_epoch`, `evaluate`, `build_dataloaders`, and every hyperparameter
  constant are now checked directly against the real `train.ipynb` (uploaded
  after this notebook's first draft) -- that draft had `loss_fn` missing from
  several calls (both functions require it) and a wrong `build_dataloaders`
  call shape; both are fixed below, and `pruning.py`/`quantization.py` had the
  same `loss_fn` bug fixed too (push the corrected versions before running
  the cells below that import them).

Same verification discipline as every other module in this project: every
self-check below runs on synthetic data and must pass BEFORE the real-data
cell beneath it runs.

**Before running:** confirm `REPO_ROOT` below actually matches this repo's
path on your Drive, and that `src/edge_fault_dx/compression/pruning.py`,
`quantization.py`, and `training/threshold_sweep.py` are the versions you
already pushed to GitHub (they're imported unchanged here, not redefined).


## 0. Setup -- mount Drive, import this project's existing, already-proven modules unchanged

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys

# TODO: confirm this matches your repo's real path on Drive
REPO_ROOT = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace"
SRC_ROOT = f"{REPO_ROOT}/src"
DATA_DIR = f"{REPO_ROOT}/data"

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)


In [ ]:
import time
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from scipy import stats

# Reused unchanged -- these must already exist at the paths below.
from edge_fault_dx.models.ma1dcnn import MA1DCNN, build_ma1dcnn, Conv1dSamePadding
from edge_fault_dx.training.train import (
    TRIAL_13_MODEL_CONFIG,
    TRIAL_13_TRAIN_CONFIG,
    DEFAULT_WINDOWED_DIR,
    N_EPOCHS,
    EARLY_STOPPING_PATIENCE,
    RANDOM_SEED,
    SCHEDULER_T_0,
    SCHEDULER_T_MULT,
    build_dataloaders,
    train_one_epoch,
    evaluate,
)

# evaluate()/train_one_epoch()'s real signatures (confirmed against train.ipynb) both
# take loss_fn as a required positional argument: train_one_epoch(model, loader,
# optimizer, loss_fn, device); evaluate(model, loader, loss_fn, device, threshold).
# run_training() itself uses nn.BCEWithLogitsLoss() -- reused unchanged here throughout.
loss_fn = nn.BCEWithLogitsLoss()
from edge_fault_dx.compression.pruning import (
    _l1_channel_importance,
    _topk_keep_indices,
    compute_pruned_channel_counts,
    recover_pruned_model,
)
from edge_fault_dx.compression.quantization import (
    quantize_model_ptq,
    quantize_model_qat,
    count_params_any,
    repair_missing_module_dicts,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# retention bar for StandardCNN, same proportional rule as MA1DCNN's
# (baseline macro-F1 - 5 points) -- filled in once Section 2 below
# actually trains standard_cnn_trial13 and prints its real baseline.
STANDARD_CNN_RETENTION_BAR = None


## 1. `models/standard_cnn.py` -- the architecture-only comparator (h1_completion_plan.docx Section 3)

**[verbatim from h1_completion_plan.docx]**, cleaned of line-wrap artifacts.
Identical to MA1DCNN except `ChannelAttentionModule` and
`ElementwiseAttentionModule` are removed -- conv3's output goes straight
into global average pooling. This is deliberate: "architecture only" is
H1's independent variable, so every other choice (channel sizes,
`kernel_size`, `fc_units`, `dropout`, training config) is copied from
`TRIAL_13_MODEL_CONFIG` / `TRIAL_13_TRAIN_CONFIG` unchanged.

Once this cell is proven against real data (Sections 2 and beyond), this
class belongs at `src/edge_fault_dx/models/standard_cnn.py` -- same
notebook-first-then-extract pattern `pruning.py`/`quantization.py` already
went through.


In [ ]:
class StandardCNN(nn.Module):
    """3 conv blocks -> global average pooling -> FC head -> 5 logits.

    Identical to MA1DCNN (models/ma1dcnn.py) except it has no
    ChannelAttentionModule and no ElementwiseAttentionModule -- conv3's
    output goes straight into global_avg_pool. This is the whole point:
    H1 isolates "architecture only" as its independent variable, so every
    other choice (channel sizes, kernel_size, fc_units, dropout, training
    config) is copied from TRIAL_13_MODEL_CONFIG / TRIAL_13_TRAIN_CONFIG
    unchanged.
    """

    def __init__(
        self,
        n_input_channels,
        channel_1,
        channel_2,
        channel_3,
        kernel_size,
        fc_units,
        dropout,
        n_outputs,
        conv_dropout=0.0,
        use_batchnorm=False,
    ):
        super().__init__()
        self.conv1 = Conv1dSamePadding(n_input_channels, channel_1, kernel_size, conv_dropout, use_batchnorm)
        self.conv2 = Conv1dSamePadding(channel_1, channel_2, kernel_size, conv_dropout, use_batchnorm)
        self.conv3 = Conv1dSamePadding(channel_2, channel_3, kernel_size, conv_dropout, use_batchnorm)
        # no self.cam, no self.eam -- this is the entire architectural difference
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(channel_3, fc_units)
        self.fc_activation = nn.ReLU()
        self.output_layer = nn.Linear(fc_units, n_outputs)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        # (MA1DCNN inserts self.cam(x) then self.eam(x) here -- the only diff)
        x = self.global_avg_pool(x)
        x = x.view(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.fc_activation(x)
        return self.output_layer(x)

    def predict_proba(self, x):
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))


def build_standard_cnn(config):
    """Same config dict shape as build_ma1dcnn() -- TRIAL_13_MODEL_CONFIG
    works unchanged as input to both, which is exactly what keeps the
    comparison fair."""
    return StandardCNN(
        n_input_channels=config["n_input_channels"],
        channel_1=config["channel_1"],
        channel_2=config["channel_2"],
        channel_3=config["channel_3"],
        kernel_size=config["kernel_size"],
        fc_units=config["fc_units"],
        dropout=config["dropout"],
        n_outputs=config["n_outputs"],
        conv_dropout=config.get("conv_dropout", 0.0),
        use_batchnorm=config.get("use_batchnorm", False),
    )


### 1.1 Self-check -- parameter count matches the hand-derived prediction

h1_completion_plan.docx Section 3's hand count: conv1+conv2+conv3 identical
to MA1DCNN (55,904), cam+eam removed (-8,481), fc+output_layer identical
(2,149) -> **58,053 total**, 12.7% smaller than MA1DCNN's confirmed 66,534.
This must match exactly before trusting the class on real data.


In [ ]:
standard_cnn = build_standard_cnn(TRIAL_13_MODEL_CONFIG)
standard_cnn_params = sum(p.numel() for p in standard_cnn.parameters())
print("StandardCNN total params:", standard_cnn_params)
assert standard_cnn_params == 58_053, (
    f"expected 58,053 per h1_completion_plan.docx Section 3, got {standard_cnn_params} -- "
    "check TRIAL_13_MODEL_CONFIG hasn't drifted from the values that document used "
    "(channel_1=32, channel_2=64, channel_3=128, kernel_size=5, fc_units=16, "
    "n_input_channels=28, n_outputs=5)."
)

# shape / probability-range self-check, same pattern ma1dcnn.ipynb used
dummy = torch.randn(4, TRIAL_13_MODEL_CONFIG["n_input_channels"], 50)
probs = standard_cnn.predict_proba(dummy)
assert probs.shape == (4, TRIAL_13_MODEL_CONFIG["n_outputs"])
assert torch.all((probs >= 0) & (probs <= 1))
print("Self-check PASSED: params == 58,053, output shape and probability range correct.")


## 2. Train the baseline (h1_completion_plan.docx Section 4)

The plan's own recommended path is a one-line `train.py` edit -- a
`build_fn=build_ma1dcnn` parameter on `run_training()` (default preserves
every existing call site). Now that `train.ipynb`'s real source is
available, that edit really is as small as the plan says: `run_training()`
(Section 9 there) hard-codes `model = build_ma1dcnn(model_config).to(device)`
as its one architecture-specific line -- adding a `build_fn=build_ma1dcnn`
parameter and using it there is the whole change.

**[new for this notebook]** `run_training_like()` below avoids editing
`train.py` directly anyway, so this notebook doesn't touch a file it
can't run end to end itself: it reuses `train_one_epoch`, `evaluate`, and
`build_dataloaders` exactly as imported, and now mirrors `run_training()`'s
real loop field-for-field (`torch.manual_seed(RANDOM_SEED)`, the same
`CosineAnnealingWarmRestarts(T_0=10, T_mult=2)`, the same
`N_EPOCHS=30` / `EARLY_STOPPING_PATIENCE=5`, the same strictly-greater
best-checkpoint rule) -- confirmed against train.ipynb Section 9, not just
assumed. If you'd rather apply the plan's literal `build_fn` edit to
`train.py` instead, do that and call
`run_training(..., build_fn=build_standard_cnn)` in place of this cell --
both now produce the same result by construction.


In [ ]:
def run_training_like(build_fn, model_config, train_config, train_loader, val_loader,
                       device, n_epochs=N_EPOCHS, patience=EARLY_STOPPING_PATIENCE):
    """[new for this notebook] Same effect as run_training(build_fn=...)
    would have, built only from the pieces already confirmed
    architecture-agnostic: build_dataloaders, train_one_epoch, evaluate.
    Mirrors run_training()'s real loop (train.ipynb Section 9)
    field-for-field: same torch.manual_seed(RANDOM_SEED), same
    CosineAnnealingWarmRestarts(T_0, T_mult), same N_EPOCHS/patience
    defaults, same strictly-greater best-checkpoint rule.
    """
    torch.manual_seed(RANDOM_SEED)
    model = build_fn(model_config).to(device)
    optimizer = Adam(model.parameters(), lr=train_config["learning_rate"],
                      weight_decay=train_config["weight_decay"])
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=SCHEDULER_T_0, T_mult=SCHEDULER_T_MULT)

    best_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, n_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics = evaluate(model, val_loader, loss_fn, device, threshold=train_config["threshold"])
        scheduler.step()
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})
        print(f"epoch {epoch:2d}  train_loss={train_loss:.4f}  val_f1_macro={val_metrics['f1_macro']:.4f}")

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"early stopping at epoch {epoch} (patience={patience})")
                break

    model.load_state_dict(best_state)
    return model, history


In [ ]:
# build_dataloaders()'s real signature (confirmed against train.ipynb):
# build_dataloaders(windowed_dir=DEFAULT_WINDOWED_DIR, batch_size=..., files=None, num_workers=0)
# -- it does NOT take a train_config dict directly.
train_loader, val_loader = build_dataloaders(
    windowed_dir=DEFAULT_WINDOWED_DIR,
    batch_size=TRIAL_13_TRAIN_CONFIG["batch_size"],
    files=None,
)

standard_cnn_trained, standard_cnn_history = run_training_like(
    build_standard_cnn,
    TRIAL_13_MODEL_CONFIG,
    TRIAL_13_TRAIN_CONFIG,
    train_loader,
    val_loader,
    device,
)

standard_cnn_baseline_f1 = max(h["f1_macro"] for h in standard_cnn_history)
print("\nstandard_cnn_trial13 best val_f1_macro:", standard_cnn_baseline_f1)

STANDARD_CNN_RETENTION_BAR = standard_cnn_baseline_f1 - 0.05
print("StandardCNN retention bar (baseline - 5 pts):", STANDARD_CNN_RETENTION_BAR)

torch.save(standard_cnn_trained, f"{DATA_DIR}/standard_cnn_trial13.pt")
print("saved:", f"{DATA_DIR}/standard_cnn_trial13.pt")


## 3. `prune_standard_cnn` (h1_completion_plan.docx Section 5.1)

**[verbatim structure from h1_completion_plan.docx]**, with the elided
`conv1 -> conv2 -> conv3` and `fc` weight-copy lines filled in from
`prune_ma1dcnn`'s own Section 5.1 logic (now in `pruning.py`) -- the plan
explicitly says this is "exactly prune_ma1dcnn's Section 5.1 ... logic,"
just without the CAM/EAM branch, since StandardCNN has neither module.


In [ ]:
def prune_standard_cnn(model, model_config, prune_ratio):
    """Structured channel pruning of conv1/conv2/conv3 by L1-norm,
    rebuilding conv2's input, conv3's input, and fc's input to match --
    the same three dependent-layer rebuilds prune_ma1dcnn does for
    MA1DCNN's conv stack, with no CAM/EAM step because StandardCNN has
    neither module (h1_completion_plan.docx Section 5.1).
    """
    c1, c2, c3 = compute_pruned_channel_counts(
        model_config["channel_1"], model_config["channel_2"], model_config["channel_3"], prune_ratio,
    )

    pruned_config = dict(model_config)
    pruned_config.update(channel_1=c1, channel_2=c2, channel_3=c3)
    pruned_model = build_standard_cnn(pruned_config)

    keep1 = _topk_keep_indices(_l1_channel_importance(model.conv1.conv.weight), c1)
    keep2 = _topk_keep_indices(_l1_channel_importance(model.conv2.conv.weight), c2)
    keep3 = _topk_keep_indices(_l1_channel_importance(model.conv3.conv.weight), c3)

    with torch.no_grad():
        pruned_model.conv1.conv.weight.copy_(model.conv1.conv.weight[keep1, :, :])
        pruned_model.conv1.conv.bias.copy_(model.conv1.conv.bias[keep1])

        pruned_model.conv2.conv.weight.copy_(model.conv2.conv.weight[keep2, :, :][:, keep1, :])
        pruned_model.conv2.conv.bias.copy_(model.conv2.conv.bias[keep2])

        pruned_model.conv3.conv.weight.copy_(model.conv3.conv.weight[keep3, :, :][:, keep2, :])
        pruned_model.conv3.conv.bias.copy_(model.conv3.conv.bias[keep3])

        # fc: Linear(channel_3, fc_units) -- in_features (columns) -> keep3, same as prune_ma1dcnn's fc step
        pruned_model.fc.weight.copy_(model.fc.weight[:, keep3])
        pruned_model.fc.bias.copy_(model.fc.bias)

        # output_layer: never pruned -- neither dimension depends on a pruned channel count
        pruned_model.output_layer.weight.copy_(model.output_layer.weight)
        pruned_model.output_layer.bias.copy_(model.output_layer.bias)

    return pruned_model


### 3.1 Self-check -- lossless at prune_ratio = 0.0 (same discipline as pruning_code_explained.docx Section 7.1)

In [ ]:
probe_model = build_standard_cnn(TRIAL_13_MODEL_CONFIG)
probe_model.eval()
probe_input = torch.randn(4, TRIAL_13_MODEL_CONFIG["n_input_channels"], 50)

with torch.no_grad():
    out_original = probe_model(probe_input)

pruned_at_zero = prune_standard_cnn(probe_model, TRIAL_13_MODEL_CONFIG, prune_ratio=0.0)
pruned_at_zero.eval()
with torch.no_grad():
    out_pruned = pruned_at_zero(probe_input)

assert torch.allclose(out_original, out_pruned, atol=1e-5), "lossless-at-ratio-0.0 check FAILED"
print("Self-check PASSED: prune_standard_cnn is lossless at prune_ratio=0.0.")


## 4. `QuantizedStandardCNNTrunk` (h1_completion_plan.docx Section 5.2)

**[verbatim structure from h1_completion_plan.docx]**, elided
`global_avg_pool` / `dropout` / `fc_activation` assignments filled in to
mirror `QuantizedConvTrunk`'s own convention exactly. Deliberately scoped
to conv1/conv2/conv3 only -- quantizing StandardCNN's FC head too, when
MA1DCNN's stays FP32, would make "how much of the model is INT8" an
uncontrolled second difference between the two pipelines (Section 5.2's
own reasoning).


In [ ]:
class QuantizedStandardCNNTrunk(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.conv1, self.conv2, self.conv3 = model.conv1, model.conv2, model.conv3
        self.dequant = torch.quantization.DeQuantStub()
        self.fc, self.output_layer = model.fc, model.output_layer   # unquantized, by reference
        self.global_avg_pool = model.global_avg_pool
        self.dropout = model.dropout
        self.fc_activation = model.fc_activation

    def forward(self, x):
        x = self.quant(x)
        x = self.conv3(self.conv2(self.conv1(x)))
        x = self.dequant(x)                       # INT8 -> FP32 before global_avg_pool
        return self.output_layer(self.fc_activation(self.fc(self.dropout(
            self.global_avg_pool(x).view(x.shape[0], -1)))))


## 5. Benchmarking -- params / size / CPU latency

**[new for this notebook]** `benchmark_variant`'s literal source isn't
quoted in any document available when this notebook was built -- only its
methodology (pruning_code_explained.docx Section 10): batch size 1, 10
warm-up runs excluded, >=100 timed runs (200 used throughout this
project), macro-F1 via `evaluate()` unchanged, params via `count_params_any`
(quantization-safe), size via a real serialized-file byte count.


In [ ]:
def benchmark_variant(model, val_loader, device, threshold, n_warmup=10, n_timed=200):
    """[new for this notebook] params / size (MB) / CPU latency p50,p95,p99 (ms),
    following pruning_code_explained.docx Section 10's documented methodology.
    """
    model.eval().to(device)

    val_metrics = evaluate(model, val_loader, loss_fn, device, threshold=threshold)

    n_params = count_params_any(model)

    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        torch.save(model, f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)
    os.unlink(f.name)

    single_input = next(iter(val_loader))[0][:1].to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            model(single_input)

        timings_ms = []
        for _ in range(n_timed):
            if device == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(single_input)
            if device == "cuda":
                torch.cuda.synchronize()
            timings_ms.append((time.perf_counter() - t0) * 1000.0)

    timings_ms = np.array(timings_ms)
    p50, p95, p99 = np.percentile(timings_ms, [50, 95, 99])

    return {
        "macro_f1": val_metrics["f1_macro"],
        "params": n_params,
        "size_mb": size_mb,
        "latency_p50_ms": p50,
        "latency_p95_ms": p95,
        "latency_p99_ms": p99,
    }


## 6. Run StandardCNN through the full Phase 4 pipeline (h1_completion_plan.docx Section 8, step 4)

Prune at 20/40/60%, recover, benchmark (Steps 1-4 equivalent) -- then PTQ,
check against `STANDARD_CNN_RETENTION_BAR`, fall back to QAT only if PTQ
misses it (the same real-result-first rule that decided PTQ-vs-QAT for
MA1DCNN, not assumed in advance).


In [ ]:
standard_cnn_results = {}

for ratio in (0.20, 0.40, 0.60):
    print(f"\n=== StandardCNN, prune_ratio={ratio:.0%} ===")

    pruned = prune_standard_cnn(standard_cnn_trained, TRIAL_13_MODEL_CONFIG, ratio)
    recovered, recovery_history = recover_pruned_model(
        pruned, TRIAL_13_TRAIN_CONFIG, train_loader, val_loader, device,
    )
    fp32_bench = benchmark_variant(recovered, val_loader, device, TRIAL_13_TRAIN_CONFIG["threshold"])
    print("FP32 recovered:", fp32_bench)

    ptq_model = quantize_model_ptq(recovered, train_loader)
    # TODO: resweep_threshold needs threshold_sweep.py's real sweep() signature --
    # confirm this call matches it once threshold_sweep.py is imported successfully.
    ptq_threshold = resweep_threshold(ptq_model, val_loader, device) if False else TRIAL_13_TRAIN_CONFIG["threshold"]
    ptq_bench = benchmark_variant(ptq_model, val_loader, device, ptq_threshold)
    print("PTQ:", ptq_bench)

    verdict = "PASSED" if ptq_bench["macro_f1"] >= STANDARD_CNN_RETENTION_BAR else "BELOW RETENTION BAR"
    print(f"PTQ verdict ({ptq_bench['macro_f1']:.4f} vs bar {STANDARD_CNN_RETENTION_BAR:.4f}): {verdict}")

    chosen = {"fp32": fp32_bench, "ptq": ptq_bench, "ratio": ratio, "method": "ptq", "recovered_model": recovered}

    if verdict == "BELOW RETENTION BAR":
        print("PTQ missed the bar -- building QAT fallback (same trigger rule as MA1DCNN's Step 5).")
        qat_model, qat_history = quantize_model_qat(
            recovered, TRIAL_13_TRAIN_CONFIG, train_loader, val_loader, device,
        )
        qat_threshold = TRIAL_13_TRAIN_CONFIG["threshold"]  # TODO: resweep once threshold_sweep.py is wired in
        qat_bench = benchmark_variant(qat_model, val_loader, device, qat_threshold)
        print("QAT:", qat_bench)
        qat_verdict = "PASSED" if qat_bench["macro_f1"] >= STANDARD_CNN_RETENTION_BAR else "BELOW RETENTION BAR"
        print(f"QAT verdict ({qat_bench['macro_f1']:.4f} vs bar {STANDARD_CNN_RETENTION_BAR:.4f}): {qat_verdict}")
        chosen.update(qat=qat_bench, method="qat" if qat_verdict == "PASSED" else "ptq")
        best_model = qat_model if qat_verdict == "PASSED" else ptq_model
    else:
        best_model = ptq_model

    chosen["best_model"] = best_model
    standard_cnn_results[ratio] = chosen

    torch.save(best_model, f"{DATA_DIR}/standard_cnn_trial13_pruned{int(ratio*100)}_quantized.pt")

print("\nStandardCNN compression pipeline complete.")
for ratio, r in standard_cnn_results.items():
    print(f"  {ratio:.0%}: method={r['method']}  macro_f1={r.get(r['method'], {}).get('macro_f1')}")


## 7. `evaluation/noise_robustness.py` (h1_completion_plan.docx Section 6)

**[verbatim from h1_completion_plan.docx]**, cleaned of line-wrap
artifacts -- this is the one piece the plan gives complete, literal source
for, since nothing in this codebase previously added evaluation-time
noise. Once proven (Section 7.1 below), this belongs at
`src/edge_fault_dx/evaluation/noise_robustness.py`.


In [ ]:
def add_snr_noise(X, snr_db, rng):
    """Adds zero-mean Gaussian noise to a window, scaled so the noisy
    signal sits at exactly snr_db decibels relative to X's own per-window
    power -- the standard SNR_dB = 10*log10(signal_power / noise_power)
    definition, inverted to solve for the noise variance that produces a
    given target SNR.

    X : np.ndarray, shape (28, 50) -- one already-normalized window
    """
    signal_power = np.mean(X ** 2)
    snr_linear = 10 ** (snr_db / 10)
    noise_power = signal_power / snr_linear
    noise = rng.normal(0.0, np.sqrt(noise_power), size=X.shape).astype(np.float32)
    return X + noise


class NoisyWindowedFaultDataset(torch.utils.data.Dataset):
    """Wraps an existing WindowedFaultDataset, adding independent
    add_snr_noise() noise to every window at a fixed SNR, with a given
    numpy Generator -- a new seed per trial is what makes repeated trials
    independent, not repeats of the identical noise draw.
    """
    def __init__(self, base_dataset, snr_db, seed):
        self.base = base_dataset
        self.snr_db = snr_db
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        X, y = self.base[idx]
        X_noisy = add_snr_noise(X.numpy(), self.snr_db, self.rng)
        return torch.from_numpy(X_noisy), y


SNR_LEVELS_DB = [0, 10, 20]      # H1's own testability spec
N_TRIALS = 5                      # H1's own minimum


def run_noise_robustness_trials(model, val_base_dataset, threshold, loss_fn, device):
    """Returns {snr_db: [trial_1_metrics, trial_2_metrics, ...]}. Clean
    metrics (no noise) are a separate baseline computed once via the
    existing evaluate() against val_base_dataset directly, kept outside
    this dict.

    NOTE: the doc's own literal signature includes loss_fn (matching
    evaluate()'s real required argument, confirmed against train.ipynb) --
    an earlier transcription of this cell dropped it while cleaning up
    the document's line-wrap artifacts; restored here.
    """
    results = {}
    for snr_db in SNR_LEVELS_DB:
        trial_metrics = []
        for trial in range(N_TRIALS):
            seed = hash((snr_db, trial)) % (2 ** 31)   # distinct, reproducible per (snr, trial)
            noisy_dataset = NoisyWindowedFaultDataset(val_base_dataset, snr_db, seed)
            noisy_loader = torch.utils.data.DataLoader(noisy_dataset, batch_size=256, shuffle=False)
            metrics = evaluate(model, noisy_loader, loss_fn, device, threshold)   # UNCHANGED function
            trial_metrics.append(metrics)
        results[snr_db] = trial_metrics
    return results


def summarize_trials(trial_metrics, metric_key="f1_macro"):
    """mean +/- 95% CI (t-distribution, small-N-appropriate for N_TRIALS=5)."""
    values = np.array([m[metric_key] for m in trial_metrics])
    mean = values.mean()
    sem = stats.sem(values)
    ci = sem * stats.t.ppf(0.975, df=len(values) - 1)
    return mean, ci


def paired_comparison(ma1dcnn_drops, standard_cnn_drops):
    """Paired t-test over matched (same snr_db, same trial index)
    accuracy-drop values -- the paired-test comparison H1's testability
    clause asks for. accuracy_drop[trial] = clean_f1_macro -
    noisy_f1_macro[trial], per model.
    """
    t_stat, p_value = stats.ttest_rel(ma1dcnn_drops, standard_cnn_drops)
    return t_stat, p_value


### 7.1 Self-check -- monotonic degradation on a synthetic/clean model

Per h1_completion_plan.docx Section 8, step 5: a model evaluated on its own
clean validation data should show monotonically falling macro-F1 as SNR
drops from 20 dB toward 0 dB, and near-chance performance at a
deliberately extreme, very low SNR -- the same "prove it standalone first"
discipline every other function in this pipeline was held to.


In [ ]:
def _noise_robustness_self_check(model, val_base_dataset, threshold, device):
    extreme_snr_db = -20   # deliberately extreme, well outside SNR_LEVELS_DB
    check_levels = [extreme_snr_db, 0, 10, 20]
    scores = {}
    for snr_db in check_levels:
        seed = hash((snr_db, "selfcheck")) % (2 ** 31)
        noisy_dataset = NoisyWindowedFaultDataset(val_base_dataset, snr_db, seed)
        noisy_loader = torch.utils.data.DataLoader(noisy_dataset, batch_size=256, shuffle=False)
        scores[snr_db] = evaluate(model, noisy_loader, loss_fn, device, threshold)["f1_macro"]
        print(f"  snr={snr_db:>4} dB  macro_f1={scores[snr_db]:.4f}")

    assert scores[0] <= scores[10] <= scores[20], (
        f"expected monotonically falling macro_f1 as SNR drops (20>=10>=0), got {scores}"
    )
    assert scores[extreme_snr_db] < scores[0], (
        f"expected near-chance degradation at extreme low SNR ({extreme_snr_db} dB) to score "
        f"below the 0 dB level, got {scores}"
    )
    print("Self-check PASSED: macro_f1 degrades monotonically as SNR drops.")

print("Running noise-robustness self-check against the recovered FP32 StandardCNN (20% ratio):")
_noise_robustness_self_check(
    standard_cnn_results[0.20]["recovered_model"],
    val_loader.dataset,
    TRIAL_13_TRAIN_CONFIG["threshold"],
    device,
)


## 8. The real paired comparison (h1_completion_plan.docx Section 8, step 6)

Pick each architecture's best-performing compressed checkpoint from
Section 6 above, run both through `run_noise_robustness_trials` at
0/10/20 dB x 5 trials, then `summarize_trials` + `paired_comparison` to
produce H1's actual reported numbers. **Set `CHOSEN_RATIO` to whichever
pruning ratio you're comparing** -- the plan doesn't name one in advance;
it's a choice you make from Section 6's real results once you have them.


In [ ]:
CHOSEN_RATIO = 0.20   # TODO: set this from your real Section 6 results

ma1dcnn_compressed = torch.load(f"{DATA_DIR}/ma1dcnn_trial13_pruned{int(CHOSEN_RATIO*100)}_quantized_qat.pt")
ma1dcnn_compressed = repair_missing_module_dicts(ma1dcnn_compressed)   # cross-process reload fix, Section 12.2

standard_cnn_compressed = standard_cnn_results[CHOSEN_RATIO]["best_model"]

clean_threshold = TRIAL_13_TRAIN_CONFIG["threshold"]

ma1dcnn_clean = evaluate(ma1dcnn_compressed, val_loader, loss_fn, device, threshold=clean_threshold)
standard_cnn_clean = evaluate(standard_cnn_compressed, val_loader, loss_fn, device, threshold=clean_threshold)
print("clean (no noise) macro_f1 -- MA1DCNN:", ma1dcnn_clean["f1_macro"], " StandardCNN:", standard_cnn_clean["f1_macro"])

ma1dcnn_noise_results = run_noise_robustness_trials(ma1dcnn_compressed, val_loader.dataset, clean_threshold, loss_fn, device)
standard_cnn_noise_results = run_noise_robustness_trials(standard_cnn_compressed, val_loader.dataset, clean_threshold, loss_fn, device)

print(f"\n{'SNR (dB)':>10} | {'MA1DCNN mean':>14} ± {'95% CI':>8} | {'StandardCNN mean':>18} ± {'95% CI':>8}")
for snr_db in SNR_LEVELS_DB:
    ma1dcnn_mean, ma1dcnn_ci = summarize_trials(ma1dcnn_noise_results[snr_db])
    standard_cnn_mean, standard_cnn_ci = summarize_trials(standard_cnn_noise_results[snr_db])
    print(f"{snr_db:>10} | {ma1dcnn_mean:>14.4f} ± {ma1dcnn_ci:>8.4f} | {standard_cnn_mean:>18.4f} ± {standard_cnn_ci:>8.4f}")

# paired accuracy-drop comparison, matched (snr_db, trial) pairs
ma1dcnn_drops = [
    ma1dcnn_clean["f1_macro"] - m["f1_macro"]
    for snr_db in SNR_LEVELS_DB for m in ma1dcnn_noise_results[snr_db]
]
standard_cnn_drops = [
    standard_cnn_clean["f1_macro"] - m["f1_macro"]
    for snr_db in SNR_LEVELS_DB for m in standard_cnn_noise_results[snr_db]
]
t_stat, p_value = paired_comparison(ma1dcnn_drops, standard_cnn_drops)
print(f"\npaired t-test on accuracy-drop (MA1DCNN vs StandardCNN): t={t_stat:.4f}, p={p_value:.4f}")


## 9. What this notebook does and does not settle (h1_completion_plan.docx Section 9)

Running Sections 1-8 for real gives a genuine, architecture-controlled
answer to whether MA1DCNN's attention modules earn their added parameters
and compute, under compression and under noise.

**What it does not settle on its own:** the uncompressed MA1DCNN baseline
(0.8401 macro-F1) is already below H1's 93.25% floor relative to
Akindoju's (2025) 98.16% -- an upstream gap this notebook does not touch
and the StandardCNN comparison cannot close, since it's a property of this
project's own confirmed baseline, not of compression or of which
architecture is compressed. And everything here runs on Colab CPU/GPU,
which is sufficient for the accuracy/precision/robustness comparison
itself, but not a substitute for the on-device confirmation H1's literal
"deployed on Jetson" wording calls for once Phase 6 exists.
